In [ ]:
# path setup: this notebook now lives in revision_scripts/, while the data paths (data/...) and the
# config / netmedgpt imports are relative to the repository root. Safe to re-run; delete if the
# notebook is moved back to the root.
import os, sys
if os.path.basename(os.getcwd()) == 'revision_scripts':
    os.chdir('..')
for p in (os.getcwd(), os.path.join(os.getcwd(), 'revision_scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)


# Walk coverage: training graph vs curated KG

Compares how much of the graph the walks cover for

1. **training walks**: the walks the benchmark models are trained on (cached corpus), measured on the training graph of each split;
2. **curated KG walks**: walks on the curated KG used for inference (`edges_preprocessed.csv`: drug-drug edges, drugs with broad indications and misclassified drug-ADR edges removed), generated with the settings of `netmedgpt.pt` (walk length 5, 30 walks per node, Node2Vec p = q = 1) and measured on the whole curated KG.

Reported: edge coverage (edges that appear in at least one walk), neighbourhood completeness (share of a node's neighbours seen next to it; average node and hubs with > 1000 neighbours), and edge coverage per relation.
No GPU, no training. Run from the repository root; results in `data/new_analysis/coverage/curated_kg/`.

In [1]:
INFERENCES = ["random_link_split"]       # the zero-shot split gives the same coverage (within 0.3 percentage points)
SEEDS = [1, 2, 3, 4, 5]                 # training splits
CURATED_NODES = "data/KG/nodes_preprocessed.csv"
CURATED_EDGES = "data/KG/edges_preprocessed.csv"
OUT_DIR = "data/new_analysis/coverage/curated_kg"
CHUNK = 1000000
HUB_DEGREE = 1001   # hubs: nodes with more than 1000 neighbours

In [2]:
# graph lookups, walk statistics and walk generation of walk_coverage_analysis.ipynb (same code, function cells only)
import json
import nbformat

FUNCTION_CELLS = ("import os", "# Key helpers", "# Training graph", "# Walk statistics", "# Walk generation", "import scipy.sparse")
loaded = 0
for cell in nbformat.read("walk_coverage_analysis.ipynb", as_version=4).cells:
    if cell.cell_type == "code" and cell.source.startswith(FUNCTION_CELLS):
        exec(cell.source)
        loaded += 1
assert loaded == len(FUNCTION_CELLS), f"found {loaded} of {len(FUNCTION_CELLS)} function cells"


# new walks with the training settings (Node2Vec, p = q = 1) on split['train_txgnn']
def generate_walk_stats(split, g):
    param = split['all_param']['node2vec']
    edge_index = torch.tensor(split['train_txgnn'][['x_index', 'y_index']].to_numpy().T, dtype=torch.long)
    set_seed(split['seed'])
    model = Node2Vec(edge_index=edge_index, embedding_dim=1, walk_length=param['walk_length'],
                     context_size=param['walk_length'], walks_per_node=param['walks_per_node'],
                     p=1.0, q=1.0, num_negative_samples=0, sparse=True)
    st = WalkStats(g)
    for pos_rw, _ in tqdm(model.loader(batch_size=param['batch_size'], shuffle=False, num_workers=param['workers']),
                          desc="walks"):
        walks = add_relations(pos_rw.numpy().astype(np.int64), g)
        if len(walks):
            st.add(walks)
    return st


def coverage_row(setting, graph, seed, g, st):
    cov = covered_undirected(g, st)
    completeness = per_node_mean(g, cov.astype(np.float64))
    return {'setting': setting, 'graph': graph, 'seed': seed, 'walks': st.n_walks, 'edges': len(cov),
            'edge_coverage_pct': 100 * cov.mean(),
            'neighbourhood_completeness_pct': 100 * np.nanmean(completeness),
            'hub_neighbourhood_completeness_pct': 100 * np.nanmean(completeness[g['degree'] >= HUB_DEGREE])}


# edge coverage per relation (share of the relation's edges whose two nodes are next to each other in a walk)
def relation_coverage(setting, graph, seed, g, st):
    cov = covered_undirected(g, st)[g['pair_und_idx'][g['t_pair_idx']]]
    df = pd.DataFrame({'relation': np.array(g['relations'])[g['t_rel']], 'covered': cov})
    out = df.groupby('relation')['covered'].agg(edges='size', edge_coverage_pct='mean').reset_index()
    out['edge_coverage_pct'] *= 100
    return out.assign(setting=setting, graph=graph, seed=seed)


# training walks of one split / seed, on its training graph
def training_coverage(split, data):
    g = build_graph(split)
    st = WalkStats(g)
    walks = data['node2vec_walks'].numpy()
    for i in range(0, len(walks), CHUNK):
        st.add(walks[i:i + CHUNK])
    graph = f"training graph ({split['inference']})"
    return coverage_row('training walks', graph, split['seed'], g, st), \
        relation_coverage('training walks', graph, split['seed'], g, st)


# new walks on the whole curated KG, one walk set per seed
def curated_coverage(nodes, edge, all_param, seeds):
    train = edge[['x_index', 'y_index', 'relation']]
    split = dict(nodes=nodes, edge=edge, train_txgnn=train, val_txgnn=train.iloc[:0], test_txgnn=train.iloc[:0],
                 all_param=all_param, seed=seeds[0])
    g = build_graph(split)
    rows, rels = [], []
    for seed in seeds:
        st = generate_walk_stats(dict(split, seed=seed), g)
        rows.append(coverage_row('curated KG walks', 'curated KG', seed, g, st))
        rels.append(relation_coverage('curated KG walks', 'curated KG', seed, g, st))
    return rows, rels

In [3]:
rows, rels = [], []

for inference in INFERENCES:
    for seed in SEEDS:
        print(f"===== training walks | {inference} | seed {seed} =====")
        split = load_split(inference, seed)
        row, rel = training_coverage(split, build_corpus(split))
        rows.append(row)
        rels.append(rel)
        del split

print("===== curated KG walks =====")
all_param = json.load(open("data/parameters.json"))
r, rl = curated_coverage(pd.read_csv(CURATED_NODES), pd.read_csv(CURATED_EDGES), all_param, SEEDS)
rows += r
rels += rl

os.makedirs(OUT_DIR, exist_ok=True)
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "coverage.csv"), index=False)
pd.concat(rels, ignore_index=True).to_csv(os.path.join(OUT_DIR, "relation_coverage.csv"), index=False)
print(f"Saved to {OUT_DIR}")

===== training walks | random_link_split | seed 1 =====
Reproducibility settings
  seed                     : 1
  torch.manual_seed        : 1
  torch.cuda.manual_seed   : 1
  numpy / random seed      : 1
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/random_link_split_seed1.pt
===== training walks | random_link_split | seed 2 =====
Reproducibility settings
  seed                     : 2
  torch.manual_seed        : 2
  torch.cuda.manual_seed   : 2
  numpy / random seed      : 2
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/random_link_split_seed2.pt
===== training walks | random_link_split | seed 3 =====
Reproducibility settings
  seed                     : 3
  torch.manual_seed        : 3
  torch.cuda.manual_seed   : 3
  numpy / random seed      : 3
  cudnn.deterministic      : True
  cudnn.benchmark          : False


/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


... (3 outputs left out)

/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])
/home/bbc8731/NetMedGPT/common.py:84: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **cfg['read_kwargs'])


Loading cached corpus: data/new_analysis/corpus/random_link_split_seed5.pt
===== curated KG walks =====


walks: 100%|████████████████████████████████████████████████████████████████████████| 125/125 [00:12<00:00, 10.10it/s]


Saved to data/new_analysis/coverage/curated_kg


## Results (mean ± std over seeds)

In [4]:
from IPython.display import display

res = pd.read_csv(os.path.join(OUT_DIR, "coverage.csv"))
cols = ['edge_coverage_pct', 'neighbourhood_completeness_pct', 'hub_neighbourhood_completeness_pct']
g = res.groupby(['setting', 'graph'], sort=False)[cols]
display(g.mean().round(1).astype(str) + " ± " + g.std().round(2).astype(str))

rel = pd.read_csv(os.path.join(OUT_DIR, "relation_coverage.csv"))
display(rel.pivot_table(index='relation', columns='graph', values='edge_coverage_pct', aggfunc='mean')
           .round(1).sort_values('curated KG'))

,,edge_coverage_pct,neighbourhood_completeness_pct,hub_neighbourhood_completeness_pct
setting,graph,,,
training walks,training graph (random_link_split),77.4 ± 0.03,97.1 ± 0.0,55.6 ± 0.06
curated KG walks,curated KG,90.8 ± 0.02,98.4 ± 0.0,88.8 ± 0.02


graph,curated KG,training graph (random_link_split)
relation,,
anatomy_protein_present,87.1,90.4
protein_protein,89.8,92.3
anatomy_protein_absent,90.4,92.9
cellcomp_protein,94.4,95.9
pathway_protein,94.7,96.3
molfunc_protein,94.8,96.2
disease_protein,97.1,96.8
bioprocess_protein,97.7,98.4
drug_effect,98.4,83.2
